# Chapter 5 — Linear Models and Regularization

**Nama:** Omar Al Farizy Hutagalung

**NIM:** 101032300034  

**Mata Kuliah:** Machine Learning  
 
**Tugas:** Tugas 2 — Code Reproduction & Theoretical Deep-Dive

---

## Objective

Chapter ini membahas linear models dan teknik regularization untuk regresi.
Materi utama meliputi Linear Regression, Ridge, Lasso, ElasticNet, serta
pengembangan regresi menggunakan polynomial features.

## 1. Introduction to Linear Models

Linear Regression mencoba menemukan hubungan linear antara fitur dan target.

Secara sederhana:

y = β₀ + β₁x₁ + β₂x₂ + ... + βₚxₚ + ε

Model mempelajari koefisien β dari data training dan kemudian menggunakan
koefisien tersebut untuk memprediksi target pada data baru.

Linear Regression menjadi dasar untuk memahami model regresi yang lebih
kompleks dan regularized regression."

In [1]:
import numpy as np

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

X, y = make_regression(
    n_samples=1000,
    n_features=100,
    n_informative=10,
    noise=20,
    random_state=123
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_test)

print("MSE:", mean_squared_error(y_test, y_pred))
print("R²:", r2_score(y_test, y_pred))

MSE: 420.67172977360286
R²: 0.9918053510489934


## 2. Ridge and Lasso Regression

Regularization menambahkan penalty pada fungsi loss untuk mengurangi
kompleksitas model dan membantu mencegah overfitting.

### Ridge — L2

Ridge memberikan penalty berdasarkan kuadrat koefisien.

Ridge cenderung mengecilkan koefisien tetapi biasanya tetap mempertahankan
semua fitur.

### Lasso — L1

Lasso memberikan penalty berdasarkan nilai absolut koefisien.

Lasso dapat membuat beberapa koefisien menjadi tepat 0 sehingga juga dapat
melakukan feature selection.

Parameter `alpha` mengontrol kekuatan regularization.

In [2]:
from sklearn.linear_model import Ridge, Lasso

ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)

lasso_model = Lasso(
    alpha=10.0,
    max_iter=10000,
    tol=0.001
)
lasso_model.fit(X_train, y_train)

ridge_pred = ridge_model.predict(X_test)
lasso_pred = lasso_model.predict(X_test)

print("Ridge")
print("MSE:", mean_squared_error(y_test, ridge_pred))
print("R²:", r2_score(y_test, ridge_pred))

print("\nLasso")
print("MSE:", mean_squared_error(y_test, lasso_pred))
print("R²:", r2_score(y_test, lasso_pred))

Ridge
MSE: 420.86987397067156
R²: 0.9918014912171537

Lasso
MSE: 1374.4662913168838
R²: 0.9732255153955899


## 3. Comparing Linear, Ridge, and Lasso

Ketiga model dapat dibandingkan menggunakan MSE dan R².

- **MSE** semakin kecil semakin baik.
- **R²** semakin mendekati 1 menunjukkan model menjelaskan variasi target
  dengan lebih baik.

Regularization tidak selalu menghasilkan peningkatan performa yang besar,
tetapi dapat membantu mengendalikan kompleksitas model, terutama ketika
fitur saling berkorelasi.

In [3]:
models = {
    "Linear": linear_model,
    "Ridge": ridge_model,
    "Lasso": lasso_model
}

for name, model in models.items():
    pred = model.predict(X_test)

    print(
        f"{name:8s} | "
        f"MSE: {mean_squared_error(y_test, pred):.2f} | "
        f"R²: {r2_score(y_test, pred):.4f}"
    )

Linear   | MSE: 420.67 | R²: 0.9918
Ridge    | MSE: 420.87 | R²: 0.9918
Lasso    | MSE: 1374.47 | R²: 0.9732


## 4. ElasticNet and Regularization

ElasticNet menggabungkan regularization L1 dan L2.

Parameter penting:

- `alpha` mengontrol kekuatan regularization.
- `l1_ratio` mengontrol keseimbangan antara L1 dan L2.

Jika `l1_ratio = 1`, perilakunya mendekati Lasso.

Jika `l1_ratio = 0`, perilakunya mendekati Ridge.

ElasticNet berguna ketika dataset memiliki fitur yang berkorelasi dan kita
tetap ingin mendapatkan efek feature selection.

In [4]:
from sklearn.linear_model import ElasticNet

elastic_model = ElasticNet(
    alpha=0.1,
    l1_ratio=0.5,
    max_iter=10000
)

elastic_model.fit(X_train, y_train)

elastic_pred = elastic_model.predict(X_test)

print("ElasticNet")
print("MSE:", mean_squared_error(y_test, elastic_pred))
print("R²:", r2_score(y_test, elastic_pred))

ElasticNet
MSE: 595.0604321437224
R²: 0.9884082741935702


## 5. Regularization Theory and Practice

Regularization berkaitan dengan bias-variance trade-off.

Tanpa regularization, model yang terlalu kompleks dapat memiliki variance
tinggi dan mengalami overfitting.

Regularization memberikan constraint pada koefisien sehingga:

- Kompleksitas model berkurang.
- Variance dapat berkurang.
- Bias dapat sedikit meningkat.
- Generalization terhadap data baru dapat menjadi lebih baik.

Perbedaan utama:

| Teknik | Penalty | Karakteristik |
|---|---|---|
| Ridge | L2 | Mengecilkan koefisien |
| Lasso | L1 | Dapat membuat koefisien menjadi 0 |
| ElasticNet | L1 + L2 | Kombinasi keduanya |

## 6. Regression and Regularization

Linear Regression hanya dapat merepresentasikan hubungan linear secara
langsung. Polynomial features dapat digunakan untuk membentuk fitur baru
sehingga model linear dapat menangkap hubungan yang lebih kompleks.

Pendekatan ini berguna ketika hubungan antara fitur dan target bersifat
curvilinear atau non-linear.

In [5]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

polynomial_model = Pipeline([
    ("polynomial", PolynomialFeatures(degree=2)),
    ("ridge", Ridge(alpha=1.0))
])

polynomial_model.fit(X_train[:, :5], y_train)

poly_pred = polynomial_model.predict(X_test[:, :5])

print("Polynomial + Ridge")
print("MSE:", mean_squared_error(y_test, poly_pred))
print("R²:", r2_score(y_test, poly_pred))

Polynomial + Ridge
MSE: 52802.367607274355
R²: -0.028585558997479277


## Conclusion

Linear Regression menjadi dasar untuk memahami hubungan antara fitur dan
target.

Ridge menggunakan L2 regularization untuk mengecilkan koefisien, sedangkan
Lasso menggunakan L1 dan dapat melakukan feature selection. ElasticNet
menggabungkan kedua pendekatan tersebut.

Regularization membantu mengendalikan kompleksitas model dan mengurangi
risiko overfitting. Polynomial features dapat digunakan ketika hubungan
antara fitur dan target tidak bersifat linear.

Pemilihan model dan tingkat regularization tetap perlu dievaluasi menggunakan
data validasi atau test serta metrik seperti MSE dan R².